# Data Privacy Brasil — publicações, temas e evidências

[Abrir no Colab pelo GitHub](https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/Mercosul/dataprivacy.ipynb) · [Fonte oficial](https://www.dataprivacybr.org/blog/)

Este notebook consulta a **API pública do Data Privacy Brasil Research**, reúne as publicações, conserva título, data, categoria, resumo e link e gera tabelas e gráficos interativos. A versão completa do agente FAPESP consegue ler a célula de coleta deste mesmo arquivo.

**Uso:** execute todas as células. Ajuste o limite de páginas, depois altere os filtros e rode novamente só a seção de gráficos, sem repetir a coleta. Ao final, baixe o ZIP com CSVs, SQLite, diagnóstico e painel HTML independente.

**Interpretação:** as categorias vêm do site. Os temas e palavras são calculados por regras sobre **títulos e resumos**, sem leitura automática dos artigos completos. Uma publicação pode aparecer em vários temas; as contagens não medem toda a produção institucional quando a cobertura da API estiver incompleta.

In [ ]:
# 1. Dependências da sessão
import importlib.util
import subprocess
import sys

PACOTES = {
    "requests": "requests>=2.32,<3",
    "bs4": "beautifulsoup4>=4.12,<5",
    "pandas": "pandas>=2.2,<3",
    "plotly": "plotly>=5.24,<7",
    "IPython": "ipython>=8,<10",
}
faltantes = [pacote for modulo, pacote in PACOTES.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)
print("Dependências prontas.")

In [ ]:
# 2. Coletor público. O agente FAPESP lê esta célula sem executar os gráficos.
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from html import unescape
from urllib.parse import urlsplit, urlunsplit
import re
import time
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

source_id = "dataprivacy"
cfg = {
    "name": "Data Privacy Brasil Research",
    "url": "https://www.dataprivacybr.org/wp-json/wp/v2/posts",
    "pages": 10,       # 100 publicações por página; para ampliar, ajuste MAX_PAGINAS abaixo
    "per_page": 100,
}


@dataclass
class Registro:
    source_id: str
    source: str
    title: str
    date: str
    url: str
    summary: str
    collected_at: str
    category: str
    categories: str
    post_id: int


def sessao():
    s = requests.Session()
    retry = Retry(total=2, backoff_factor=0.5,
                  status_forcelist=(429, 500, 502, 503, 504),
                  allowed_methods=("GET",))
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; FAPESP-Research-Scraper/1.0; academic use)"})
    return s


def texto_html(value):
    return re.sub(r"\s+", " ", unescape(BeautifulSoup(value or "", "html.parser").get_text(" ", strip=True))).strip()


def url_oficial(value):
    parsed = urlsplit(value or "")
    if parsed.scheme not in {"http", "https"} or parsed.hostname not in {"dataprivacybr.org", "www.dataprivacybr.org"}:
        return ""
    return urlunsplit(("https", "www.dataprivacybr.org", parsed.path.rstrip("/") + "/", "", ""))


def collect(source_id: str, cfg: dict, delay: float = 0.35):
    """Return records and coverage diagnostics; a missing page is never counted as zero news."""
    s = sessao()
    categories_url = "https://www.dataprivacybr.org/wp-json/wp/v2/categories"
    category_names, warnings = {}, []
    try:
        response = s.get(categories_url, params={"per_page": 100, "_fields": "id,name"}, timeout=30)
        response.raise_for_status()
        category_names = {int(item["id"]): texto_html(item.get("name", "")) for item in response.json()}
    except (requests.RequestException, ValueError, KeyError) as exc:
        warnings.append("Categorias indisponíveis: " + str(exc)[:160])

    seen, errors, pages_ok, attempted = {}, [], 0, 0
    limit = max(1, int(cfg.get("pages", 1)))
    site_pages = None
    for page in range(1, limit + 1):
        if site_pages is not None and page > site_pages:
            break
        attempted += 1
        params = {"per_page": min(100, max(1, int(cfg.get("per_page", 100)))),
                  "page": page, "orderby": "date", "order": "desc",
                  "_fields": "id,date,link,title,excerpt,categories"}
        try:
            response = s.get(cfg["url"], params=params, timeout=35)
            response.raise_for_status()
            items = response.json()
            if not isinstance(items, list):
                raise ValueError("API não retornou uma lista de publicações")
            if page == 1:
                site_pages = int(response.headers.get("X-WP-TotalPages", "0")) or None
            pages_ok += 1
            for item in items:
                title = texto_html(item.get("title", {}).get("rendered", ""))
                url = url_oficial(item.get("link", ""))
                date = str(item.get("date", ""))[:10]
                if not title or not url:
                    continue
                if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", date):
                    date = ""
                cats = [category_names.get(int(v), f"Categoria {v}") for v in item.get("categories", [])]
                now = datetime.now(timezone.utc).isoformat()
                seen[url] = Registro(source_id, cfg["name"], title, date, url,
                                     texto_html(item.get("excerpt", {}).get("rendered", "")),
                                     now, cats[0] if cats else "Sem categoria",
                                     " | ".join(cats) if cats else "Sem categoria", int(item["id"]))
            if not items:
                break
        except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
            errors.append({"page": page, "error": f"{type(exc).__name__}: {exc}"[:250]})
            break
        if page < limit and (site_pages is None or page < site_pages):
            time.sleep(delay)

    status = "partial" if errors and seen else "failed" if errors else "ok" if seen else "empty"
    diagnostic = {"source_id": source_id, "source": cfg["name"], "status": status,
                  "pages_ok": pages_ok, "pages_requested": attempted,
                  "site_total_pages": site_pages, "records": len(seen),
                  "errors": errors, "warnings": warnings,
                  "coverage": "API pública de posts; cada página contém até 100 publicações"}
    return [asdict(row) for row in seen.values()], diagnostic

## 3. Coleta e qualidade

A API informa o total de páginas. O limite abaixo evita varreduras sem fim. Uma página que falha aparece no diagnóstico, e o histórico da rodada não é apresentado como completo.

In [ ]:
#@title 3. Coletar publicações
MAX_PAGINAS = 10 #@param {type:"integer"}
if not 1 <= MAX_PAGINAS <= 100:
    raise ValueError("MAX_PAGINAS deve estar entre 1 e 100.")

from pathlib import Path
import pandas as pd
from IPython.display import display

configuracao = dict(cfg, pages=MAX_PAGINAS)
registros, diagnostico = collect(source_id, configuracao)
if not registros:
    raise RuntimeError("Nenhuma publicação foi coletada. Confira o diagnóstico: " + str(diagnostico))

df = pd.DataFrame(registros).drop_duplicates(subset=["url"]).copy()
df["data_dt"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["data_dt", "title"], ascending=[False, True]).reset_index(drop=True)
resumo_cobertura = pd.DataFrame([{
    "fonte": diagnostico["source"], "estado": diagnostico["status"],
    "páginas lidas": diagnostico["pages_ok"],
    "páginas do site": diagnostico["site_total_pages"],
    "URLs únicas": len(df), "datas válidas": int(df["data_dt"].notna().sum()),
    "última publicação": df["data_dt"].max().date().isoformat() if df["data_dt"].notna().any() else "—",
}])
display(resumo_cobertura)
display(df[["date", "title", "category", "url"]].head(15))
if diagnostico["errors"] or diagnostico["warnings"]:
    display(pd.DataFrame(diagnostico["errors"] + [{"warning": x} for x in diagnostico["warnings"]]))

## 4. Classificação transparente

Os cinco temas correspondem às perguntas da pesquisa FAPESP. As regras procuram expressões no título e no resumo publicados pelo site. Elas servem para localizar evidências para leitura qualitativa; **não** classificam automaticamente o conteúdo integral nem indicam influência normativa.

In [ ]:
import unicodedata
from collections import Counter

def normalizar(value):
    plain = unicodedata.normalize("NFKD", str(value).lower())
    return "".join(ch for ch in plain if not unicodedata.combining(ch))

TEMAS = {
    "Dados e LGPD": r"\blgpd\b|dados pessoais|protecao de dados|privacidade|anpd|gdpr",
    "IA e algoritmos": r"inteligencia artificial|\bia\b|algoritm|sistema automatiz|deepfake",
    "Plataformas": r"plataform|moderacao|redes sociais|conteudo digital|tiktok|discord|meta\b|google\b",
    "Infância e idade": r"crianc|adolescen|infancia|afericao de idade|verificacao de idade|eca digital",
    "Governança global": r"sul global|mercosul|\bbrics\b|\bg20\b|\bonu\b|governanca global|uniao europeia|regulacao internacional",
}
STOPWORDS = set("a ao aos as com como da das de do dos e em entre esse essa esta este foi na nas no nos o os ou para pela pelo por que se sem sobre sua suas seu seus um uma umas uns mais menos ainda tambem brasil data privacy blog artigo noticia publicacao".split())

df["texto_analise"] = (df["title"].fillna("") + " " + df["summary"].fillna("")).map(normalizar)
for tema, pattern in TEMAS.items():
    df[tema] = df["texto_analise"].str.contains(pattern, regex=True, na=False)
df["temas"] = df.apply(lambda row: " | ".join(tema for tema in TEMAS if row[tema]) or "Nenhum dos cinco", axis=1)
df["mes"] = df["data_dt"].dt.to_period("M").astype(str)
df.loc[df["data_dt"].isna(), "mes"] = "Sem data"
df["ano"] = df["data_dt"].dt.year.astype("Int64")

def frequencia_titulos(frame, limit=20):
    words = Counter()
    for title in frame["title"].fillna(""):
        words.update({word for word in re.findall(r"[a-z]{4,}", normalizar(title))
                      if word not in STOPWORDS})
    return pd.DataFrame(words.most_common(limit), columns=["palavra", "títulos"])

display(df["temas"].value_counts().rename_axis("temas por título/resumo").reset_index(name="publicações"))
print("As categorias do site e os temas da pesquisa podem se sobrepor.")

## 5. Painel interativo

Altere os filtros e execute apenas a célula abaixo. Para pesquisar uma categoria ou um tema, digite parte do nome. O período considera a data de publicação. A tabela final contém links para conferir cada item na fonte.

In [ ]:
#@title 5. Filtrar e comparar
PERIODO = "Todo o histórico" #@param ["Últimos 30 dias", "Últimos 90 dias", "Último ano", "Todo o histórico"]
CATEGORIA = "" #@param {type:"string"}
TEMA = "" #@param {type:"string"}
BUSCA = "" #@param {type:"string"}

import plotly.express as px
import plotly.graph_objects as go
from datetime import date as data_hoje
from IPython.display import HTML

visao = df.copy()
dias = {"Últimos 30 dias": 30, "Últimos 90 dias": 90,
        "Último ano": 365, "Todo o histórico": None}[PERIODO]
if dias is not None:
    limite = pd.Timestamp(data_hoje.today()) - pd.Timedelta(days=dias - 1)
    visao = visao[visao["data_dt"] >= limite]
if CATEGORIA.strip():
    visao = visao[visao["categories"].map(normalizar).str.contains(re.escape(normalizar(CATEGORIA.strip())), na=False)]
if TEMA.strip():
    opcoes = [x for x in TEMAS if normalizar(TEMA.strip()) in normalizar(x)]
    if len(opcoes) != 1:
        raise ValueError(f"Escolha um tema entre: {list(TEMAS)}")
    visao = visao[visao[opcoes[0]]]
if BUSCA.strip():
    visao = visao[visao["texto_analise"].str.contains(re.escape(normalizar(BUSCA.strip())), na=False)]
visao = visao.copy()
print(f"Seleção: {len(visao)} de {len(df)} publicações; {visao['data_dt'].notna().sum()} com data.")
FIGURAS = []
if visao.empty:
    print("Nenhum registro satisfaz os filtros. Altere a seleção e execute esta célula novamente.")
else:
    mensal = visao.dropna(subset=["data_dt"]).groupby("mes").size().reset_index(name="publicações")
    if not mensal.empty:
        f = px.bar(mensal, x="mes", y="publicações", title="Publicações coletadas por mês")
        FIGURAS.append(f)

    categorias = visao["category"].value_counts().head(12).rename_axis("categoria").reset_index(name="publicações")
    FIGURAS.append(px.bar(categorias, x="publicações", y="categoria", orientation="h",
                          title="Categorias do site — categoria principal"))

    temas = pd.DataFrame({"tema": list(TEMAS), "publicações": [int(visao[x].sum()) for x in TEMAS]})
    FIGURAS.append(px.bar(temas, x="tema", y="publicações", title="Temas citados em títulos e resumos"))

    serie = visao.dropna(subset=["data_dt"])[["mes", *TEMAS]].melt(
        id_vars="mes", var_name="tema", value_name="presente")
    if not serie.empty:
        serie = serie.groupby(["mes", "tema"], as_index=False)["presente"].sum()
        FIGURAS.append(px.line(serie, x="mes", y="presente", color="tema", markers=True,
                               title="Evolução mensal das menções temáticas"))

    termos = frequencia_titulos(visao)
    if not termos.empty:
        FIGURAS.append(px.bar(termos, x="títulos", y="palavra", orientation="h",
                               title="Palavras mais frequentes nos títulos (um voto por título)"))

    maiores = visao["category"].value_counts()
    maiores = maiores[maiores >= 5].head(10).index.tolist()
    if maiores:
        matriz = visao[visao["category"].isin(maiores)].groupby("category")[list(TEMAS)].mean().mul(100).round(1)
        FIGURAS.append(px.imshow(matriz, x=list(matriz.columns), y=list(matriz.index),
                                labels={"color": "% dos itens"}, color_continuous_scale="Blues",
                                aspect="auto", text_auto=True,
                                title="Menções por categoria (% dentro de cada categoria; n ≥ 5)"))

    semana = visao.dropna(subset=["data_dt"]).copy()
    if not semana.empty:
        dias_semana = {0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
        semana["dia"] = semana["data_dt"].dt.dayofweek.map(dias_semana)
        distribuicao = semana["dia"].value_counts().reindex(dias_semana.values(), fill_value=0).rename_axis("dia").reset_index(name="publicações")
        FIGURAS.append(px.bar(distribuicao, x="dia", y="publicações", title="Dia da semana de publicação"))

    for figura in FIGURAS:
        figura.update_layout(template="plotly_white", height=460,
                             font=dict(family="Arial", size=13), margin=dict(l=55, r=35, t=70, b=75))
        figura.show()
    evidencias = visao[["date", "category", "title", "temas", "url"]].head(40)
    display(HTML(evidencias.to_html(index=False, escape=True, render_links=True)))
    print("Cada linha da tabela pode ser conferida pelo link da publicação original.")

## 6. Exportação

O CSV completo mantém as URLs e os diagnósticos da coleta. O painel HTML inclui o código do Plotly para abrir localmente, mesmo sem conexão. A base SQLite contém a mesma tabela principal para futuras análises.

In [ ]:
#@title 6. Salvar CSV, SQLite, HTML e ZIP
BAIXAR_ZIP_AUTOMATICAMENTE = False #@param {type:"boolean"}
import json
import shutil
import sqlite3
from datetime import datetime
from IPython.display import FileLink

SAIDA = Path("/content/fapesp-data/dataprivacy") if Path("/content").exists() else Path.cwd() / "fapesp-data/dataprivacy"
SAIDA.mkdir(parents=True, exist_ok=True)
colunas = ["post_id", "date", "title", "category", "categories", "summary", "url", "temas", "collected_at"]
df[colunas].to_csv(SAIDA / "publicacoes_data_privacy_brasil.csv", index=False, encoding="utf-8-sig")
visao[colunas].to_csv(SAIDA / "selecao_filtrada.csv", index=False, encoding="utf-8-sig")
frequencia_titulos(visao).to_csv(SAIDA / "palavras_titulos.csv", index=False, encoding="utf-8-sig")
(SAIDA / "diagnostico.json").write_text(json.dumps(diagnostico, ensure_ascii=False, indent=2), encoding="utf-8")
with sqlite3.connect(SAIDA / "data_privacy_brasil.sqlite") as con:
    df[colunas].to_sql("publicacoes", con, if_exists="replace", index=False)

html_parts = ["<!doctype html><html lang='pt-BR'><meta charset='utf-8'>",
              "<meta name='viewport' content='width=device-width, initial-scale=1'>",
              "<title>Data Privacy Brasil — painel FAPESP</title>",
              "<style>body{font:16px Arial,sans-serif;max-width:1250px;margin:30px auto;padding:0 20px;color:#243247}"
              "h1{color:#173b73}table{border-collapse:collapse;display:block;overflow-x:auto}"
              "td,th{border:1px solid #ddd;padding:7px;text-align:left}tr:nth-child(even){background:#f5f7fa}</style>",
              "<h1>Data Privacy Brasil — publicações</h1>",
              f"<p>Gerado em {datetime.now().strftime('%d/%m/%Y %H:%M')}. "
              f"{len(visao)} publicações na seleção; {diagnostico['pages_ok']}/{diagnostico['pages_requested']} páginas lidas. "
              "Temas obtidos de títulos e resumos, com links para verificação.</p>"]
for i, figura in enumerate(FIGURAS):
    html_parts.append(figura.to_html(full_html=False, include_plotlyjs=(i == 0)))
html_parts.append("<h2>Registros recentes da seleção</h2>")
html_parts.append(visao[["date", "category", "title", "temas", "url"]].head(80).to_html(
    index=False, escape=True, render_links=True))
html_parts.append("</html>")
(SAIDA / "painel_interativo.html").write_text("\n".join(html_parts), encoding="utf-8")
ZIP = Path(shutil.make_archive(str(SAIDA.parent / ("DataPrivacyBrasil_" + datetime.now().strftime("%Y%m%d"))), "zip", SAIDA))
print("Arquivos:", *(str(p) for p in sorted(SAIDA.iterdir())), sep="\n- ")
display(FileLink(str(ZIP)))
if BAIXAR_ZIP_AUTOMATICAMENTE:
    try:
        from google.colab import files
        files.download(str(ZIP))
    except ImportError:
        print("O download automático só está disponível no Colab; o ZIP está no caminho acima.")